# 문제1

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
df = pd.read_csv("sleep_fatigue.csv")

# 피쳐 및 타겟 분리
features = ["avg_sleep_hours", "night_shift_days", "caffeine_intake"]
target = "fatigue_score"

X = df[features]
y = df[target]

# StandardScaler 적용
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 학습-평가 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 모델 학습
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 평가 (RMSE 계산)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("rmse : ", rmse)

# 예측값 저장
df_result = df.loc[y_test.index, ["soldier_id"]].copy()
df_result["actual_score"] = y_test.values
df_result["predicted_score"] = y_pred
result_path = "fatigue_pred.csv"
df_result.to_csv(result_path, index=False)

# 예측-실제 시각화
plt.figure(figsize=(8, 6))
plt.plot(y_test.values, label="Actual", marker='o')
plt.plot(y_pred, label="Predicted", marker='x')
plt.xlabel("Test Sample Index")
plt.ylabel("Fatigue Score")
plt.title("Actual vs Predicted Fatigue Score")
plt.legend()
plt.grid(True)
plot_path = "fatigue_lineplot.png"
plt.savefig(plot_path)
plt.close()

---

# 문제2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import torch
import cv2
import os

# YOLOv5 모델 로드 (torch.hub)
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model.classes = [0]  # 클래스 0: person

# 메타데이터 로드
df_meta = pd.read_csv("event_meta.csv")

# 이미지 경로 설정
image_dir = "./images/unit_events"
image_files = ['event_001.png', 'event_002.png', 'event_003.png']

# 탐지 및 집계
unit_counts = defaultdict(list)

for _, row in df_meta.iterrows():
    img_path = os.path.join(image_dir, row["filename"])

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(str(img_path))[:, :, ::-1]  # BGR to RGB
    results = model(img)
    df_result = results.pandas().xyxy[0]
    person_count = (df_result['name'] == 'person').sum()
    unit_counts[row["unit_name"]].append(person_count)

# 부대별 통계 요약
summary = []
for unit, counts in unit_counts.items():
    summary.append({
        "unit_name": unit,
        "total_count": sum(counts),
        "average_count": round(sum(counts) / len(counts), 2)
    })

# 결과 저장
df_summary = pd.DataFrame(summary).sort_values(by="total_count", ascending=False)
df_summary.to_csv("event_people_stats.csv", index=False, encoding='utf-8-sig')

# 시각화
plt.figure(figsize=(8, 6))
plt.bar(df_summary["unit_name"], df_summary["total_count"], color="cornflowerblue")
plt.title("Person Count By Unit")
plt.xlabel("Unit")
plt.ylabel("Total Person Count")
plt.tight_layout()
plt.savefig("top_units.png")
plt.show()

---

# 문제3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import numpy as np

# 데이터 로드
df = pd.read_csv("service_survey.csv")

# 피처/타겟 분리
X = df[["leadership_score", "facility_score", "meal_score", "rest_time_score"]]
y = df["overall_satisfaction"]

# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 성능 평가
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

# 예측 결과 저장
df_result = df.loc[y_test.index, ["soldier_id"]].copy()
df_result["actual"] = y_test.values
df_result["predicted"] = y_pred
result_path = "satisfaction_pred.csv"
df_result.to_csv(result_path, index=False)

# 변수 중요도 시각화
importances = model.feature_importances_
feature_names = X.columns
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 6))
plt.bar([feature_names[i] for i in sorted_idx], importances[sorted_idx], color="skyblue")
plt.title("Feature Importance")
plt.ylabel("Importance Score")
plt.tight_layout()
importance_path = "importance.png"
plt.savefig(importance_path)
plt.close()

# 상위 2개 변수 산점도
top2_features = [feature_names[i] for i in sorted_idx[:2]]

plt.figure(figsize=(8, 6))
plt.scatter(X[top2_features[0]], X[top2_features[1]], c=y, cmap="viridis", edgecolor="k", alpha=0.7)
plt.colorbar(label="Satisfaction")
plt.xlabel(top2_features[0])
plt.ylabel(top2_features[1])
plt.title("Top 2 Features Scatter Plot")
plt.tight_layout()
scatter_path = "top2_scatter.png"
plt.savefig(scatter_path)
plt.close()